## LightGBM hyperparameter search on representative tickers

This notebook searches for robust LightGBM hyperparameters on a small set of representative `test_tickers`, using the existing walk-forward cross-validation (`WFCV`) pipeline.


In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import ParameterGrid

try:
    from lightgbm import LGBMRegressor
except ImportError as e:
    raise ImportError(
        "lightgbm n'est pas installe. Installe-le avec `pip install lightgbm` puis relance."
    ) from e

from utils import extract, Close_price, log_return, feature_engineering_rf, WFCV


COLS_TO_DROP = [
    'KOCRD Index', 'NOBRON Index', 'CABROVER Index', 'EB03/3M Index', 'BZSTSETA Index',
    'MXNZ0EN Index', 'EGDIMW Index', 'UX1 Index', 'VXEEM Index', 'VXEFA Index', 'VXEWZ Index',
    'VXFXI Index', 'VXGDX Index', 'VXSLV Index', 'W 1 Comdty', 'W1DOW Index', 'WBI Index',
    'FNRETTM Index', 'FNREU Index', 'WIG Index', 'WIGCON Index', 'WIGFOOD Index', 'XU100 Index',
    'XUHIZ Index', 'XUMAL Index', 'XUSIN Index', 'XUTEK Index', 'YW1 Comdty', 'ZAREUR Curncy',
    'ZARJPY Curncy', 'ZARUSD Curncy'
]

TEST_TICKERS = [
    'GC1 Comdty',
    'BLQ2TRUU Index',
    'GBPJPY Curncy',
    'M4EUHDVD Index',
    'HFRXM Index',
    'GB1M Index',
    'QW2I Index',
    'FNER Index',
    'SPX Index',
]

df = pd.read_csv('Datasets/returns_all.csv')
df.drop(columns=[c for c in COLS_TO_DROP if c in df.columns], inplace=True)
all_tickers = df.columns[1:].tolist()
tickers_name = 'all_tickers'

print(f'{len(all_tickers)} tickers disponibles apres filtrage.')
print('Test tickers:', TEST_TICKERS)

1035 tickers disponibles apres filtrage.
Test tickers: ['GC1 Comdty', 'BLQ2TRUU Index', 'GBPJPY Curncy', 'M4EUHDVD Index', 'HFRXM Index', 'GB1M Index', 'QW2I Index', 'FNER Index', 'SPX Index']


In [3]:
def make_lgbm_model(params):
    base_params = {
        'objective': 'regression',
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': -1,
    }
    base_params.update(params)
    return LGBMRegressor(**base_params)


def stats_forecasting_lgbm(df_all, name_ticker, model_params, feature_engineering=feature_engineering_rf, step_size=50, fold_size=200):
    df_ticker = extract(df_all, name_ticker)
    df_ticker['Close'] = Close_price(df_ticker)
    df_ticker['log_return'] = log_return(df_ticker)
    df_ticker = feature_engineering(df_ticker)

    X = df_ticker.drop(columns=['return', 'log_return', 'Close'])
    y = df_ticker['log_return']

    model = make_lgbm_model(model_params)
    y_pred, y_truth, mse_tab, r2 = WFCV(X, y, model, step_size=step_size, fold_size=fold_size)
    reg = sm.OLS(y_truth, sm.add_constant(y_pred)).fit()

    return {
        'SYMBOL': name_ticker,
        'MSE': float(np.mean(mse_tab)) if len(mse_tab) else np.nan,
        'OLS_R2': float(reg.rsquared),
        'OLS_Intercept': float(reg.params[0]),
        'OLS_Slope': float(reg.params[1]),
        'OLS_P_Value_Intercept': float(reg.pvalues[0]),
        'P_Value_Slope': float(reg.pvalues[1]),
        'n_samples': int(len(df_ticker)),
    }


def evaluate_lgbm_config(df_all, tickers, model_params, feature_engineering=feature_engineering_rf, step_size=50, fold_size=200):
    per_ticker_results = []
    for ticker in tickers:
        result = stats_forecasting_lgbm(
            df_all=df_all,
            name_ticker=ticker,
            model_params=model_params,
            feature_engineering=feature_engineering,
            step_size=step_size,
            fold_size=fold_size,
        )
        per_ticker_results.append(result)

    df_results = pd.DataFrame(per_ticker_results)

    summary = {
        'mean_mse': float(df_results['MSE'].mean()),
        'median_mse': float(df_results['MSE'].median()),
        'std_mse': float(df_results['MSE'].std()),
        'mean_ols_r2': float(df_results['OLS_R2'].mean()),
        'median_ols_r2': float(df_results['OLS_R2'].median()),
        'n_tickers': int(len(df_results)),
    }

    return df_results, summary


def search_lgbm_hyperparams(df_all, tickers, param_grid, feature_engineering=feature_engineering_rf, step_size=50, fold_size=200):
    search_rows = []
    per_config_results = {}

    for i, params in enumerate(ParameterGrid(param_grid), start=1):
        df_results, summary = evaluate_lgbm_config(
            df_all=df_all,
            tickers=tickers,
            model_params=params,
            feature_engineering=feature_engineering,
            step_size=step_size,
            fold_size=fold_size,
        )

        row = {'config_id': i}
        row.update(params)
        row.update(summary)
        search_rows.append(row)
        per_config_results[i] = {
            'params': params,
            'per_ticker_results': df_results,
            'summary': summary,
        }

        print(f"Config {i}: mean_mse={summary['mean_mse']:.6f} | mean_ols_r2={summary['mean_ols_r2']:.4f} | params={params}")

    search_df = pd.DataFrame(search_rows).sort_values(['mean_mse', 'median_mse']).reset_index(drop=True)
    return search_df, per_config_results


In [3]:
# Starter grid for a first-pass search.
# You can widen or narrow it depending on runtime.
param_grid = {
    'n_estimators': [200, 500],
    'learning_rate': [0.02, 0.05],
    'num_leaves': [15, 31, 63],
    'max_depth': [-1, 5, 8],
    'min_child_samples': [20, 50],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'reg_alpha': [0.0, 0.1],
    'reg_lambda': [1.0, 3.0],
}

STEP_SIZE = 50
FOLD_SIZE = 200

search_df, per_config_results = search_lgbm_hyperparams(
    df_all=df,
    tickers=TEST_TICKERS,
    param_grid=param_grid,
    feature_engineering=feature_engineering_rf,
    step_size=STEP_SIZE,
    fold_size=FOLD_SIZE,
)

display(search_df.head(10))

# Best configuration and detailed per-ticker scores
best_config_id = int(search_df.iloc[0]['config_id'])
best_params = per_config_results[best_config_id]['params']
best_per_ticker = per_config_results[best_config_id]['per_ticker_results']

print('Best config id:', best_config_id)
print('Best params:', best_params)
display(best_per_ticker)



c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


Config 1: mean_mse=0.000293 | mean_ols_r2=0.0502 | params={'colsample_bytree': 0.8, 'learning_rate': 0.02, 'max_depth': -1, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 15, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'subsample': 0.8}


c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


Config 2: mean_mse=0.000292 | mean_ols_r2=0.0503 | params={'colsample_bytree': 0.8, 'learning_rate': 0.02, 'max_depth': -1, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 15, 'reg_alpha': 0.0, 'reg_lambda': 3.0, 'subsample': 0.8}


c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


...

c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


Config 288: mean_mse=0.000318 | mean_ols_r2=0.0224 | params={'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 8, 'min_child_samples': 50, 'n_estimators': 500, 'num_leaves': 63, 'reg_alpha': 0.1, 'reg_lambda': 3.0, 'subsample': 0.8}


,config_id,colsample_bytree,learning_rate,max_depth,min_child_samples,n_estimators,num_leaves,reg_alpha,reg_lambda,subsample,mean_mse,median_mse,std_mse,mean_ols_r2,median_ols_r2,n_tickers
0,28,0.8,0.02,-1,50,200,15,0.1,3.0,0.8,0.00028,0.000121,0.000537,0.021322,0.014763,9
1,32,0.8,0.02,-1,50,200,31,0.1,3.0,0.8,0.00028,0.000121,0.000537,0.021322,0.014763,9
2,36,0.8,0.02,-1,50,200,63,0.1,3.0,0.8,0.00028,0.000121,0.000537,0.021322,0.014763,9
3,76,0.8,0.02,5,50,200,15,0.1,3.0,0.8,0.00028,0.000121,0.000537,0.021322,0.014763,9
4,80,0.8,0.02,5,50,200,31,0.1,3.0,0.8,0.00028,0.000121,0.000537,0.021322,0.014763,9
5,84,0.8,0.02,5,50,200,63,0.1,3.0,0.8,0.00028,0.000121,0.000537,0.021322,0.014763,9
6,124,0.8,0.02,8,50,200,15,0.1,3.0,0.8,0.00028,0.000121,0.000537,0.021322,0.014763,9
7,128,0.8,0.02,8,50,200,31,0.1,3.0,0.8,0.00028,0.000121,0.000537,0.021322,0.014763,9
8,132,0.8,0.02,8,50,200,63,0.1,3.0,0.8,0.00028,0.000121,0.000537,0.021322,0.014763,9
9,26,0.8,0.02,-1,50,200,15,0.0,3.0,0.8,0.00028,0.000119,0.000538,0.041011,0.039608,9


Best config id: 28
Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.02, 'max_depth': -1, 'min_child_samples': 50, 'n_estimators': 200, 'num_leaves': 15, 'reg_alpha': 0.1, 'reg_lambda': 3.0, 'subsample': 0.8}


,SYMBOL,MSE,OLS_R2,OLS_Intercept,OLS_Slope,OLS_P_Value_Intercept,P_Value_Slope,n_samples
0,GC1 Comdty,0.000121,0.035097,0.000234,0.983318,0.135078,2.466002e-40,5175
1,BLQ2TRUU Index,0.000041,0.013150,0.000032,0.895443,0.793871,1.632517e-09,2981
2,GBPJPY Curncy,0.000055,0.014763,-0.000002,0.746275,0.983027,8.316961e-23,6711
3,M4EUHDVD Index,0.000156,0.023853,0.000236,0.748926,0.098241,3.857205e-34,6367
4,HFRXM Index,0.000017,0.006459,0.000016,0.738586,0.785369,1.834883e-07,4450
5,GB1M Index,0.001693,0.007675,0.000122,0.352679,0.919126,2.839745e-11,5991
6,QW2I Index,0.000003,0.000713,0.000073,0.324902,0.006583,3.266681e-02,6617
7,FNER Index,0.000281,0.046319,0.000199,0.809745,0.354291,2.115423e-66,6459
8,SPX Index,0.000152,0.043866,0.000202,0.978471,0.177851,6.649148e-63,6479


In [ ]:

# Optional exports
# search_df.to_csv('Resultats/lgbm_hyperparam_search_test_tickers.csv', index=False)
# best_per_ticker.to_csv('Resultats/lgbm_best_config_test_tickers_detail.csv', index=False)

In [4]:
# Run best params on ALL tickers
BEST_PARAMS = {
    "colsample_bytree": 0.8,
    "learning_rate": 0.02,
    "max_depth": 5,
    "min_child_samples": 50,
    "n_estimators": 200,
    "num_leaves": 15,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "subsample": 0.8,
}
STEP_SIZE = 50
FOLD_SIZE = 200
results_all = []
for ticker in all_tickers:
    results_all.append(
        stats_forecasting_lgbm(
            df_all=df,
            name_ticker=ticker,
            model_params=BEST_PARAMS,
            feature_engineering=feature_engineering_rf,
            step_size=STEP_SIZE,
            fold_size=FOLD_SIZE,
        )
    )
df_lgbm_all = pd.DataFrame(results_all)
model_name = "LGBM_opti"
csv_name = f"Resultats/resultats_all_tickers_{model_name}.csv"
# Sauvegarde
df_lgbm_all.to_csv(csv_name, index=False)
display(df_lgbm_all.head())
print("Saved to:", csv_name)

c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pa

,SYMBOL,MSE,OLS_R2,OLS_Intercept,OLS_Slope,OLS_P_Value_Intercept,P_Value_Slope,n_samples
0,ADBF Index,0.000277,0.018154,0.000487,0.461741,0.089188,5.096172e-13,3061
1,ADCM Index,0.000528,0.069549,0.000220,0.760662,0.592385,2.075710e-43,2861
2,ADCT Index,0.000318,0.047996,0.000159,0.747637,0.596556,2.584879e-32,3054
3,ADEG Index,0.000799,0.032957,0.000134,0.663028,0.777170,4.539762e-21,2888
4,ADHC Index,0.000981,0.034551,0.001611,0.740987,0.301555,7.286464e-05,692


Saved to: Resultats/resultats_all_tickers_LGBM_opti.csv
